# Fraud Detection

Notebook ini membuat workflow sederhana untuk mendeteksi transaksi fraud dengan data transaksi saja.

Fokus utama:
- data cleaning dan preprocessing
- handling missing values
- handling class imbalance
- feature engineering sederhana
- baseline model dan LightGBM
- hyperparameter tuning dengan Optuna
- experiment tracking dengan MLflow
- pembuatan `submission.csv`

EDA dibuat minimal supaya notebook tetap terlihat seperti tugas implementasi model, bukan eksplorasi kompetisi Kaggle yang terlalu panjang.


In [1]:
!pip install -q optuna mlflow

import os
import warnings
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import optuna
import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
N_TRIALS = 20
MISSING_THRESHOLD = 0.90


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.3 MB/s eta 0:00:00


## 1. Load Dataset

Dataset yang dipakai hanya `train_transaction.csv` dan `test_transaction.csv` dari folder `dataset/`.

File submission akhir dibuat manual dari `TransactionID` pada data test.


In [2]:
DATA_DIR = Path("/kaggle/input/datasets/farisaryaputra/fraud-detection/")
train_path = DATA_DIR / "train_transaction.csv"
test_path = DATA_DIR / "test_transaction.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "File dataset tidak ditemukan. Pastikan ada dataset/train_transaction.csv dan dataset/test_transaction.csv."
    )

print(f"Dataset folder: {DATA_DIR.resolve()}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head()


Dataset folder: /kaggle/input/datasets/farisaryaputra/fraud-detection
Train shape: (590540, 394)
Test shape : (506691, 393)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,...,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,NaN,NaN,NaN,0.0,T,T,T,M2,...,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,M0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,315.0,NaN,NaN,NaN,315.0,T,T,T,M0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,25.0,1.0,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,84.0,NaN,NaN,NaN,NaN,111.0,NaN,NaN,NaN,M0,...,1.0,1.0,1.0,1.0,38.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,50.0,1758.0,925.0,0.0,354.0,0.0,135.0,0.0,0.0,0.0,50.0,1404.0,790.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Ringkasan Data Minimal

Bagian ini hanya mengecek ukuran data, missing value, dan distribusi target. Ini cukup untuk memahami masalah tanpa EDA panjang.


In [3]:
fraud_distribution = train["isFraud"].value_counts(normalize=True).rename("ratio")
print("Distribusi target:")
print(fraud_distribution)

missing_summary = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .rename("missing_rate")
)

print("\nTop 10 missing value columns:")
print(missing_summary)


Distribusi target:
isFraud
0    0.96501
1    0.03499
Name: ratio, dtype: float64

Top 10 missing value columns:
dist2    0.936284
D7       0.934099
D13      0.895093
D14      0.894695
D12      0.890410
D6       0.876068
D9       0.873123
D8       0.873123
V153     0.861237
V149     0.861237
Name: missing_rate, dtype: float64


## 3. Feature Engineering Sederhana

Feature yang dibuat sengaja sederhana dan mudah dijelaskan:
- log dari nominal transaksi
- hari dan jam transaksi dari `TransactionDT`
- bagian utama dan suffix dari domain email


In [4]:
def add_simple_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "TransactionAmt" in df.columns:
        df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    if "TransactionDT" in df.columns:
        seconds_per_day = 24 * 60 * 60
        seconds_per_hour = 60 * 60
        df["transaction_day"] = (df["TransactionDT"] // seconds_per_day).astype("int16")
        df["transaction_hour"] = ((df["TransactionDT"] // seconds_per_hour) % 24).astype("int8")

    for col in ["P_emaildomain", "R_emaildomain"]:
        if col in df.columns:
            email = df[col].astype("string").fillna("Unknown")
            parts = email.str.split(".", n=2, expand=True)
            df[f"{col}_main"] = parts[0].fillna("Unknown")
            if parts.shape[1] > 1:
                df[f"{col}_suffix"] = parts[1].fillna("Unknown")
            else:
                df[f"{col}_suffix"] = "Unknown"

    return df

train = add_simple_features(train)
test = add_simple_features(test)

print("Train shape after feature engineering:", train.shape)
print("Test shape after feature engineering :", test.shape)


Train shape after feature engineering: (590540, 401)
Test shape after feature engineering : (506691, 400)


## 4. Feature Selection dan Split Data

Kolom dengan missing value terlalu tinggi dibuang. `TransactionID`, target, dan `TransactionDT` tidak digunakan sebagai fitur langsung.

Split menggunakan stratify supaya proporsi fraud dan non-fraud tetap seimbang antara training dan validation.


In [5]:
target = "isFraud"
id_col = "TransactionID"
exclude_cols = {id_col, target, "TransactionDT"}

all_feature_cols = [col for col in train.columns if col not in exclude_cols]

train_missing = train[all_feature_cols].isna().mean()
test_missing = test[all_feature_cols].isna().mean()

high_missing_cols = set(train_missing[train_missing > MISSING_THRESHOLD].index)
high_missing_cols.update(test_missing[test_missing > MISSING_THRESHOLD].index)

constant_cols = {
    col for col in all_feature_cols
    if train[col].nunique(dropna=False) <= 1
}

drop_cols = high_missing_cols | constant_cols
feature_cols = [col for col in all_feature_cols if col not in drop_cols]

print(f"Total initial features : {len(all_feature_cols)}")
print(f"Dropped high missing   : {len(high_missing_cols)}")
print(f"Dropped constant       : {len(constant_cols)}")
print(f"Final features         : {len(feature_cols)}")

X = train[feature_cols]
y = train[target].astype(int)
X_test_raw = test[feature_cols]

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train_raw.shape)
print("X_valid:", X_valid_raw.shape)


Total initial features : 398
Dropped high missing   : 2
Dropped constant       : 0
Final features         : 396
X_train: (472432, 396)
X_valid: (118108, 396)


## 5. Preprocessing

Numerical features diisi dengan median. Categorical features diisi dengan `Unknown`, lalu diubah menjadi angka menggunakan `OrdinalEncoder`.

Encoder dan nilai median hanya di-fit dari training split agar tidak terjadi data leakage ke validation set.


In [6]:
def fit_preprocessor(X_fit: pd.DataFrame) -> dict:
    numeric_cols = X_fit.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [col for col in X_fit.columns if col not in numeric_cols]

    numeric_medians = (
        X_fit[numeric_cols]
        .replace([np.inf, -np.inf], np.nan)
        .median()
        .fillna(0)
    )

    encoder = None
    if categorical_cols:
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )
        encoder.fit(X_fit[categorical_cols].fillna("Unknown").astype(str))

    return {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_medians": numeric_medians,
        "encoder": encoder,
    }


def transform_features(X_data: pd.DataFrame, preprocessor: dict) -> pd.DataFrame:
    numeric_cols = preprocessor["numeric_cols"]
    categorical_cols = preprocessor["categorical_cols"]
    numeric_medians = preprocessor["numeric_medians"]
    encoder = preprocessor["encoder"]

    parts = []

    if numeric_cols:
        numeric_part = (
            X_data[numeric_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(numeric_medians)
        )
        parts.append(numeric_part.astype("float32"))

    if categorical_cols:
        categorical_values = encoder.transform(
            X_data[categorical_cols].fillna("Unknown").astype(str)
        )
        categorical_part = pd.DataFrame(
            categorical_values,
            columns=categorical_cols,
            index=X_data.index,
        ).astype("float32")
        parts.append(categorical_part)

    processed = pd.concat(parts, axis=1)
    return processed[X_data.columns].astype("float32")

preprocessor = fit_preprocessor(X_train_raw)
X_train = transform_features(X_train_raw, preprocessor)
X_valid = transform_features(X_valid_raw, preprocessor)

print("Processed train shape:", X_train.shape)
print("Processed valid shape:", X_valid.shape)
print("Any object dtype left:", len(X_train.select_dtypes(include="object").columns))


Processed train shape: (472432, 396)
Processed valid shape: (118108, 396)
Any object dtype left: 0


## 6. Helper Evaluasi dan MLflow

Metrik utama adalah ROC-AUC karena data fraud biasanya sangat imbalance. Precision, recall, F1, accuracy, dan confusion matrix tetap disimpan sebagai tambahan.


In [7]:
MLFLOW_TRACKING_DIR = Path("mlruns").resolve()
mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.as_uri())
mlflow.set_experiment("fraud_detection_uts")


def evaluate_predictions(model_name: str, y_true, y_prob, threshold: float = 0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "model": model_name,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def log_confusion_matrix(y_true, y_prob, model_name: str, threshold: float = 0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Non Fraud", "Fraud"],
        yticklabels=["Non Fraud", "Fraud"],
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix - {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()

    with tempfile.TemporaryDirectory() as tmp_dir:
        artifact_path = Path(tmp_dir) / f"{model_name}_confusion_matrix.png"
        fig.savefig(artifact_path, dpi=120)
        mlflow.log_artifact(str(artifact_path), artifact_path="confusion_matrix")

    plt.close(fig)


def log_metrics_dict(metrics_dict: dict):
    for key, value in metrics_dict.items():
        if key != "model":
            mlflow.log_metric(key, float(value))


2026/05/12 18:19:56 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_uts' does not exist. Creating a new experiment.


## 7. Baseline Model: Logistic Regression

Baseline dibuat sederhana menggunakan Logistic Regression dengan `class_weight="balanced"` untuk membantu menangani class imbalance.


In [8]:
baseline_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=300,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

with mlflow.start_run(run_name="baseline_logistic_regression"):
    mlflow.log_param("model_name", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("feature_count", X_train.shape[1])

    baseline_model.fit(X_train, y_train)
    baseline_prob = baseline_model.predict_proba(X_valid)[:, 1]

    baseline_metrics = evaluate_predictions(
        "LogisticRegression", y_valid, baseline_prob
    )
    log_metrics_dict(baseline_metrics)
    log_confusion_matrix(y_valid, baseline_prob, "logistic_regression")
    mlflow.sklearn.log_model(baseline_model, artifact_path="model")

baseline_metrics


2026/05/12 18:21:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:21:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model': 'LogisticRegression',
 'roc_auc': np.float64(0.8635992320914162),
 'precision': 0.13297031442121904,
 'recall': 0.7326397290104041,
 'f1': 0.22508827355510128,
 'accuracy': 0.8234751244623565}

## 8. LightGBM + Optuna

LightGBM dipakai sebagai model utama karena kuat untuk data tabular. Imbalance ditangani dengan `scale_pos_weight`.

Optuna digunakan untuk mencari hyperparameter yang lebih baik berdasarkan ROC-AUC validation.


In [9]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("scale_pos_weight:", round(scale_pos_weight, 2))


def objective(trial: optuna.Trial) -> float:
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
        "scale_pos_weight": scale_pos_weight,
        "num_leaves": trial.suggest_int("num_leaves", 31, 180),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "subsample": trial.suggest_float("subsample", 0.70, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
    }

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_param("trial_number", trial.number)

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        valid_prob = model.predict_proba(X_valid)[:, 1]
        trial_metrics = evaluate_predictions("LightGBM", y_valid, valid_prob)
        log_metrics_dict(trial_metrics)

    return trial_metrics["roc_auc"]


study = optuna.create_study(direction="maximize")

with mlflow.start_run(run_name="lightgbm_optuna_tuning"):
    mlflow.log_param("n_trials", N_TRIALS)
    mlflow.log_param("feature_count", X_train.shape[1])
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    study.optimize(objective, n_trials=N_TRIALS)

    mlflow.log_metric("best_roc_auc", study.best_value)
    mlflow.log_params({f"best_{key}": value for key, value in study.best_params.items()})

print("Best ROC-AUC:", study.best_value)
print("Best params:")
study.best_params


[I 2026-05-12 18:21:27,212] A new study created in memory with name: no-name-f2b7d84f-5e89-4284-b9ef-62f58522c4d3


Negative samples: 455902
Positive samples: 16530
scale_pos_weight: 27.58


[I 2026-05-12 18:23:25,831] Trial 0 finished with value: 0.9430674734946767 and parameters: {'num_leaves': 139, 'max_depth': 9, 'learning_rate': 0.010052457541352837, 'n_estimators': 728, 'subsample': 0.8382040104326365, 'colsample_bytree': 0.6172422705494687, 'reg_alpha': 1.7824826708146588, 'reg_lambda': 0.07914583818857568}. Best is trial 0 with value: 0.9430674734946767.
[I 2026-05-12 18:25:41,859] Trial 1 finished with value: 0.9614246484262284 and parameters: {'num_leaves': 57, 'max_depth': 10, 'learning_rate': 0.024978264493234655, 'n_estimators': 1347, 'subsample': 0.7869248379631415, 'colsample_bytree': 0.8226426461129837, 'reg_alpha': 0.9571895179220726, 'reg_lambda': 1.9721394798219616}. Best is trial 1 with value: 0.9614246484262284.
[I 2026-05-12 18:26:10,911] Trial 2 finished with value: 0.9070970872153028 and parameters: {'num_leaves': 168, 'max_depth': 4, 'learning_rate': 0.04214695397552148, 'n_estimators': 300, 'subsample': 0.7767369742733901, 'colsample_bytree': 0.91

Best ROC-AUC: 0.9689044788316444
Best params:


{'num_leaves': 71,
 'max_depth': 8,
 'learning_rate': 0.11713363830465376,
 'n_estimators': 1053,
 'subsample': 0.9567559026255533,
 'colsample_bytree': 0.8841816415851622,
 'reg_alpha': 1.5888547077694488,
 'reg_lambda': 0.5906529829584772}

## 9. Evaluasi Best LightGBM

Model terbaik dari Optuna dilatih ulang pada training split, lalu dievaluasi pada validation split.


In [10]:
best_lgbm_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
    "scale_pos_weight": scale_pos_weight,
    **study.best_params,
}

best_lgbm = lgb.LGBMClassifier(**best_lgbm_params)

with mlflow.start_run(run_name="best_lightgbm_validation"):
    mlflow.log_params(best_lgbm_params)
    mlflow.log_param("feature_count", X_train.shape[1])

    best_lgbm.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    best_lgbm_prob = best_lgbm.predict_proba(X_valid)[:, 1]
    best_lgbm_metrics = evaluate_predictions("LightGBM + Optuna", y_valid, best_lgbm_prob)

    log_metrics_dict(best_lgbm_metrics)
    log_confusion_matrix(y_valid, best_lgbm_prob, "best_lightgbm")
    mlflow.sklearn.log_model(best_lgbm, artifact_path="model")

results = pd.DataFrame([baseline_metrics, best_lgbm_metrics])
results.sort_values("roc_auc", ascending=False)


2026/05/12 18:55:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:55:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,model,roc_auc,precision,recall,f1,accuracy
1,LightGBM + Optuna,0.968904,0.618712,0.806436,0.700210,0.975836
0,LogisticRegression,0.863599,0.132970,0.732640,0.225088,0.823475


## 10. Train Final Model dan Buat Submission

Setelah model terbaik dipilih, preprocessing di-fit ulang menggunakan seluruh data training. Model final kemudian dilatih dengan seluruh data training dan dipakai untuk memprediksi data test.


In [11]:
full_preprocessor = fit_preprocessor(X)
X_full = transform_features(X, full_preprocessor)
X_test_final = transform_features(X_test_raw, full_preprocessor)

y_full = y.copy()
full_negative_count = int((y_full == 0).sum())
full_positive_count = int((y_full == 1).sum())
full_scale_pos_weight = full_negative_count / full_positive_count

final_params = {
    **best_lgbm_params,
    "scale_pos_weight": full_scale_pos_weight,
}

final_model = lgb.LGBMClassifier(**final_params)

with mlflow.start_run(run_name="final_lightgbm_submission"):
    mlflow.log_params(final_params)
    mlflow.log_param("training_rows", X_full.shape[0])
    mlflow.log_param("feature_count", X_full.shape[1])

    final_model.fit(X_full, y_full)
    test_prob = final_model.predict_proba(X_test_final)[:, 1]

    final_submission = test[["TransactionID"]].copy()
    final_submission["isFraud"] = test_prob
    final_submission.to_csv("submission.csv", index=False)

    mlflow.sklearn.log_model(final_model, artifact_path="model")
    mlflow.log_artifact("submission.csv", artifact_path="submission")

print("submission.csv created")
final_submission.head()


2026/05/12 18:58:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:58:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


submission.csv created


,TransactionID,isFraud
0,3663549,0.000667
1,3663550,0.000397
2,3663551,0.000194
3,3663552,0.001279
4,3663553,0.000268


## 11. Kesimpulan Singkat

- Dataset fraud memiliki class imbalance, sehingga metrik ROC-AUC lebih tepat dibanding accuracy saja.
- Logistic Regression dipakai sebagai baseline sederhana.
- LightGBM dengan Optuna digunakan sebagai model utama karena cocok untuk data tabular dengan banyak fitur.
- MLflow menyimpan parameter, metrik, model, confusion matrix, dan file submission.

Untuk membuka MLflow UI secara lokal, jalankan command berikut di terminal:

```bash
mlflow ui --backend-store-uri mlruns
```
